# 05 · Workflows and agents

**Session 01 · Building AI applications for research and data · notebook 5 of 6**

## Objective

Notebook 04 gave the model one tool and one question at a time. The main question of this session needs more than that: figures for two countries, which come from the tool, and context from the documents, which comes from the search of notebook 03. Someone has to decide which calls to make, and in what order. This notebook shows two ways of deciding, runs both on the main question, and then looks at what an answer built from several calls still lacks.

The application so far can search a store of documents (notebook 03) and call a tool for exact numbers (notebook 04). Here the search becomes a second tool, next to the first, and the application gains the means to coordinate them.

Four terms, in plain words:

- A **workflow** is a fixed order of steps written in code: search, then look up, then write. Every run makes the same calls.
- An **agent** is a model that chooses its own next step. It asks for a tool, sees the result, and decides what to do next, until it can answer. The agent here is built with LangGraph, a library for loops of this kind.
- A **trace** is the list of every call the agent made and what came back.
- A **step limit** (`recursion_limit` in LangGraph) is the number of steps after which a run is stopped, so that a loop that never ends cannot run for ever. A step is one turn of the loop: the model's turn, or the tools' turn.

## The application after this notebook

![Diagram: the question goes to a model that chooses the next step. It can call search_documents, the search of notebook 03 now offered as a tool, which returns passages, or get_indicator, the tool of notebook 04, which returns data. The results go back to the model, which writes the answer. New in this notebook: the model chooses the steps, a step limit and a trace bound and record the loop, and the answer ends on the question of whether its numbers can be trusted, which notebook 06 takes up.]()

## What you will see

Seven short scenes:

1. A workflow answering the main question, with each step printed as it runs.
2. An agent answering the same question: the same calls, chosen by the model, shown as a trace table.
3. A question that the tools cannot answer, and what the agent does.
4. A call that fails, and how the agent recovers.
5. A loop that never ends, stopped by the step limit.
6. A comparison: when a workflow is the better design and when an agent is.
7. A look at the answer: it mixes numbers from tools, passages and the model's own arithmetic. Can we trust them? The question for notebook 06.

## How to run it

Run the cells in order. The model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. The searches, the tool results, the tables and the checks are computed when you run the cell. The exercises at the end run without a key.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
import json
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

docs = w.load_documents()
store = w.build_store(w.split_sections(docs))   # the store of notebooks 02 and 03
search_documents = w.make_search_tool(store, k=3, since="2020")   # the search of notebook 03, offered as a tool
tools = [search_documents, w.get_indicator]   # the tool of notebook 04 is the second one

INSTRUCTIONS = """You are a research assistant for development data.
Use get_indicator for every statistic and search_documents for definitions, context, and explanations.
Name the indicator and the source with each figure; cite passages in the square-bracket form given.
If the tools do not provide what is needed, say so. Do not quote figures from memory."""

## 1. A workflow: the order is written in code

The picture contrasts the two designs. In a workflow, the steps are written in code in advance: search the documents, look up the four series, then ask the model to write. The model is called once, at the end, and every run makes the same calls. The cell runs the workflow on the main question and prints each step as it happens.

![Diagram: in a workflow the order of the calls is written in code: search the documents, look up the four series, the model writes the answer, the same on every run. In an agent the model decides the next step: it asks for search_documents or get_indicator, gets the result back, and answers in text when it has enough; a step limit stops a loop that never ends and a trace lists every call.]()

In [3]:
compose = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS), ("human", "PASSAGES:\n{passages}\n\nDATA:\n{data}\n\nQUESTION: {question}")]) | model


def fixed_workflow(question):
    passages = search_documents.invoke({"query": question})
    print(f"step 1: searched the documents, found {len(passages)} passages")
    series = [w.indicator_lookup(i, c, start=2010) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
    print(f"step 2: looked up {len(series)} series: internet use and GDP per capita, for Kenya and Rwanda")
    answer = compose.invoke({
        "passages": "\n\n".join(f"{p['citation']}\n{p['text']}" for p in passages),
        "data": json.dumps(series),
        "question": question,
    })
    print("step 3: the model wrote the answer\n")
    return answer, series


w.use("nb05_compose")
workflow_answer, workflow_series = fixed_workflow(w.MAIN_QUESTION)
w.show(workflow_answer)

step 1: searched the documents, found 3 passages
step 2: looked up 4 series: internet use and GDP per capita, for Kenya and Rwanda
step 3: the model wrote the answer

[recorded output: nb05_compose]
Internet use reached 35.0 percent of the population in Kenya and 31.7 percent in Rwanda in 2024
(World Development Indicators, IT.NET.USER.ZS). GDP per capita in current US dollars was 2,133 in
Kenya and 1,060 in Rwanda in the same year (NY.GDP.PCAP.CD). Since 2010, internet use has risen by
about 28 percentage points in Kenya and 24 in Rwanda; GDP per capita has roughly doubled in Kenya
and risen by about 80 percent in Rwanda. The retrieved briefs attribute Kenya's 2023 dip in dollar
GDP per capita to a currency depreciation [kenya-economy · GDP per capita · 2026] and note that
Rwanda's growth exceeded 7 percent in most years [rwanda-economy · Growth · 2026].


The application can now answer the main question with a fixed set of steps. It is predictable: the same question makes the same calls, so it is easy to test.

## 2. An agent: the model chooses the steps

In an agent, the model is offered the tools and their descriptions, and chooses. `create_agent` builds the loop with LangGraph: the model replies with the calls it wants, the loop runs them, the results go back to the model, and the loop stops when the model answers in text. The cell asks the same question, then shows the trace as a table, one row per call, grouped by round.

In [4]:
agent = create_agent(model, tools, system_prompt=INSTRUCTIONS)

w.use("nb05_agent_main")
result = agent.invoke({"messages": [HumanMessage(content=w.MAIN_QUESTION)]}, config={"recursion_limit": 10})
w.show_trace(result)
print()
w.show(w.final_text(result))

round,call,tool,what was asked,what came back
1,1,search_documents,query=internet use and GDP per capita trends Kenya Rwanda,"3 passages, first [rwanda-digital · Internet use · 2026]"
1,2,get_indicator,"indicator=IT.NET.USER.ZS, country=Kenya, start=2010, end=2024","15 observations, 2010 to 2024, latest 35.0"
1,3,get_indicator,"indicator=IT.NET.USER.ZS, country=Rwanda, start=2010, end=2024","15 observations, 2010 to 2024, latest 31.7"
1,4,get_indicator,"indicator=NY.GDP.PCAP.CD, country=Kenya, start=2010, end=2024","15 observations, 2010 to 2024, latest 2,133.5"
1,5,get_indicator,"indicator=NY.GDP.PCAP.CD, country=Rwanda, start=2010, end=2024","15 observations, 2010 to 2024, latest 1,059.9"



Internet use. The share of individuals using the internet reached 35.0 percent of the population in
Kenya and 31.7 percent in Rwanda in 2024, up from 7.2 and 8.0 percent in 2010 (World Development
Indicators, IT.NET.USER.ZS).

Economic development. GDP per capita in current US dollars was 2,133 in Kenya and 1,060 in Rwanda in
2024, up from 1,092 and 587 in 2010 (NY.GDP.PCAP.CD). Kenya's figure dipped to 1,943 in 2023, which
the retrieved brief attributes to a currency depreciation [kenya-economy · GDP per capita · 2026].

Patterns. Kenya has the higher level on both indicators; Rwanda has the faster growth on both.
Internet use converged to about a third of the population in both countries, while Kenya's GDP per
capita remained about twice Rwanda's.

Sources: World Development Indicators via the data tool; the retrieved briefs for context.


The model chose the same five calls that the workflow made. On a different question it would choose differently. That is the point of an agent, and also its risk: the calls are not known in advance.

The first column of the table is the round: one message from the model, which can ask for several tools at once. Here it asked for all five in its first message, so they share round 1; in section 4, a call that fails takes a second round to correct. The application can now let the model choose between its tools.

## 3. A question the tools cannot answer

A good agent searches, finds nothing usable, and says so. The question is about coffee exports. The collection holds one note on coffee that carries no figures, and the data tool has no series on exports.

In [5]:
w.use("nb05_agent_offscope")
result = agent.invoke({"messages": [HumanMessage(content="How have Kenya's coffee export earnings changed in recent years?")]}, config={"recursion_limit": 10})
w.show_trace(result)
print()
w.show(w.final_text(result))

round,call,tool,what was asked,what came back
1,1,search_documents,query=Kenya coffee export earnings recent years,"3 passages, first [coffee · intro · 2026]"



The available material does not support an answer with figures. The document collection has one note
on coffee in East Africa, which describes the sector in general terms and states that it carries no
figures [coffee · intro · 2026]. The data tool holds five indicators (internet use, mobile
subscriptions, GDP per capita, GDP growth, population) and no series on coffee exports. To answer
this question, a trade or agricultural statistics source would be needed.


The agent made one search, read what came back, and declined to invent figures. The application can now say that it cannot answer, and what would be needed.

## 4. A call that fails

The model asks for an indicator id that does not exist. The tool does not crash: it returns an error as data, with the list of valid ids. The model reads it and tries again. Returning errors as data is what lets the loop recover.

In [6]:
w.use("nb05_agent_error")
result = agent.invoke({"messages": [HumanMessage(content="What share of Rwanda's population used the internet in 2023?")]}, config={"recursion_limit": 10})
w.show_trace(result)
print()
w.show(w.final_text(result))

round,call,tool,what was asked,what came back
1,1,get_indicator,"indicator=IT.NET.USERS.ZS, country=Rwanda, start=2023, end=2023",error: unknown indicator 'IT.NET.USERS.ZS'
2,2,get_indicator,"indicator=IT.NET.USER.ZS, country=Rwanda, start=2023, end=2023","1 observations, 2023 to 2023, latest 28.8"



In 2023, 28.8 percent of Rwanda's population used the internet (World Development Indicators,
IT.NET.USER.ZS).


The first row is the failed call, in round 1; the second is the retry that worked, in round 2. The application can now recover from a wrong call, as long as its tools return errors that the model can read.

## 5. A loop that never ends

A model can also keep calling tools without ever answering. The step limit stops it: after the given number of steps, the run ends with an error (`GraphRecursionError`), and the application decides what to do next. It can answer with what it has, ask the user, or give up. The recorded scenario below is a model that never stops: the cell sets the limit to 6, and the model keeps asking for tools until the loop is cut off.

In [7]:
from langgraph.errors import GraphRecursionError

w.use("nb05_runaway")
try:
    agent.invoke({"messages": [HumanMessage(content=w.MAIN_QUESTION)]}, config={"recursion_limit": 6})
except GraphRecursionError as e:
    print("stopped:", str(e)[:80])
    print("The application now chooses: here, it tells the user that it could not finish.")

stopped: Recursion limit of 6 reached without hitting a stop condition. You can increase 
The application now chooses: here, it tells the user that it could not finish.


The application can now stop a loop that does not end, instead of waiting for it.

## 6. Choosing between the two

| | Workflow | Agent |
|---|---|---|
| Calls per run | Known in advance | Varies with the question |
| Behaviour on a new kind of question | Same calls, possibly irrelevant | Adapts, possibly wrongly |
| Cost | Predictable | Bounded by the step limit |
| Testing | Test the final step and the data | Test the trace and the answer |
| Use when | The questions have one shape | The questions vary and the tools are few and safe |

The worked example has one shape, so a workflow is the right design for it. The agent earns its place when users ask questions the designer did not anticipate, and only with a step limit, a trace, and tools that fail as data.

## 7. Can we trust the numbers?

Both designs produce an answer that holds several numbers. Some come straight from a tool, some come from a passage, and some are worked out by the model itself: "has risen by about 28 percentage points" is a subtraction, which no tool returned. The answer is plain text, and a reader, or a program, has no way to tell which number is which.

The tools have already laid some groundwork. Every figure the data tool returns carries a label, called a `claim_id`, made of the indicator, the country and the year. The cell shows the latest figure of each series that the workflow fetched.

In [8]:
latest = [(s["indicator"], s["country"], o["claim_id"], f"{o['value']:,.1f}" if o["value"] < 1000 else f"{o['value']:,.0f}") for s in workflow_series for o in s["observations"][-1:]]
w.show_table(["indicator", "country", "claim_id", "value"], latest, source="Every figure the tool returns has a claim_id: indicator, country, year.")

indicator,country,claim_id,value
IT.NET.USER.ZS,KEN,IT.NET.USER.ZS:KEN:2024,35.0
IT.NET.USER.ZS,RWA,IT.NET.USER.ZS:RWA:2024,31.7
NY.GDP.PCAP.CD,KEN,NY.GDP.PCAP.CD:KEN:2024,"2,133"
NY.GDP.PCAP.CD,RWA,NY.GDP.PCAP.CD:RWA:2024,"1,060"


The labels sit on the tool's side, and nothing in the answer points back to them. What if every number in an answer carried its label, and a program checked each number against the record it names? The program has a protocol for exactly that, **Proof-Carrying Numbers** (PCN), its approach to trustworthy data, and notebook 06 builds it into the application.

## Key points

- A workflow is a fixed order of steps written in code: predictable and easy to test. Prefer it when the questions have one shape.
- An agent is a model that chooses its next step. It adapts to the question, so it needs a step limit, a trace, and tools that return their errors as data.
- Both designs need the same from their tools: clear descriptions and results the model can read.
- An answer built from several calls mixes numbers that come from tools, from passages and from the model's own arithmetic, and plain text does not say which is which. The labels the tool already attaches are the groundwork for checking that, in notebook 06.

## Exercise

1. In section 2, change `recursion_limit` from 10 to 3 and run the cell, then try 4. What is the smallest limit that lets this question finish? What does the application see when the limit is too low?
2. After section 2, run `sum(len(m.tool_calls) for m in result["messages"] if getattr(m, "tool_calls", None))`. How many calls did the agent make, and how many did the workflow make in section 1? Why are the two numbers equal here, and when would they differ?
3. Run `w.indicator_lookup("IT.NET.USER.ZS", "Uganda", start=2010)["observations"]`. The tool returns an empty list. What would the workflow of section 1 do with this result, and what would the agent do?

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [06 · Making the application trustworthy](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/06-verification-and-evaluation.ipynb)